# Matrix-weighted localization in $SE(3)$

This notebook estimates a short sequence of three-dimensional poses from observations of known landmarks and relative-pose measurements. Each landmark observation has a full anisotropic information matrix: errors perpendicular to the observation ray are weighted more strongly than errors along the ray.

The reusable [`MatrixWeightedLocalizationExample.py`](MatrixWeightedLocalizationExample.py) dataset follows the GTSAM convention: ground-truth poses are $wTk$, mapping frame $k$ into the world. The matrix-weighted factors use the opposite convention and optimize $kTw=(wTk)^{-1}$. This notebook makes every conversion between those conventions explicit.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/MatrixWeightedLocalizationExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [11]:
import numpy as np
import gtsam
import plotly.graph_objects as go
from MatrixWeightedLocalizationExample import (
    ANISOTROPICITY,
    LATERAL_SIGMA,
    ODOMETRY_SIGMA,
    MatrixWeightedLocalization,
)
from gtsam.symbol_shorthand import X

## Matrix-weighted measurement model

The dataset stores conventional GTSAM poses $wTk$. Measurements are generated directly from them: $kP=wTk.\operatorname{transformTo}(wL)$ and $iTj=(wTi)^{-1}wTj$.

The matrix-weighted graph instead uses $kTw = (kRw, k\mathbf{t}_w) \in SE(3)$ as its unknown. For a known world-frame landmark $wL_l$ and its measured sensor-frame position $\widetilde{kP}_{kl}$, the residual is

$$e_{kl}(kTw) = kRw\,wL_l + k\mathbf{t}_w - \widetilde{kP}_{kl}.$$

The corresponding factor cost is

$$\frac{1}{2}e_{kl}^{\mathsf T}W_{kl}e_{kl},$$

where $W_{kl}=\Sigma_{kl}^{-1}$ is a full information matrix. The landmark is known data, so each landmark observation is a unary factor on one pose. Known landmarks anchor the trajectory; no separate pose prior is needed.

For the controlled anisotropic model, let $u$ be the unit observation ray, $P_r=uu^{\mathsf T}$ the radial projector, and $P_l=I-uu^{\mathsf T}$ the lateral projector. We choose lateral standard deviation $\sigma_l$ and radial standard deviation $\sigma_r=\rho\sigma_l$, where $\rho$ is the anisotropicity. Then

$$\Sigma=\sigma_l^2P_l+\sigma_r^2P_r,$$

and therefore

$$W=\frac{1}{\sigma_l^2}I+\left(\frac{1}{\sigma_r^2}-\frac{1}{\sigma_l^2}\right)uu^{\mathsf T}.$$

Thus $\rho=\sigma_r/\sigma_l=\sqrt{\operatorname{cond}(\Sigma)}$. With $\rho=10$, radial noise has ten times the standard deviation and one hundred times the variance of lateral noise. For the graph variables $iTw=(wTi)^{-1}$ and $jTw=(wTj)^{-1}$, the conventional GTSAM relative pose $iTj=(wTi)^{-1}wTj$ satisfies $iTw=iTj\,jTw$. The left-composed residual is therefore $\operatorname{vec}(iTw-iTj\,jTw)$.

In [12]:
example = MatrixWeightedLocalization(num_poses=3)
measurements = example.exact_measurements()
radial_sigma = ANISOTROPICITY * LATERAL_SIGMA

print(f"sigma_l={LATERAL_SIGMA:.3f} m, sigma_r={radial_sigma:.3f} m")
print(f"sqrt(cond(Sigma))={ANISOTROPICITY:.1f}")

sigma_l=0.010 m, sigma_r=0.100 m
sqrt(cond(Sigma))=10.0


## Build the factor graph

Every pose observes all four noncoplanar landmarks. We use the dataset's `exact_measurements()` method here; `perturbed_measurements()` provides deterministic noisy measurements without changing the stored $wTk$ ground truth.

Before inserting ground truth into the graph, we invert each $wTk$ because the matrix-weighted factors optimize $kTw$. The landmark measurements $kP$ and relative measurements $iTj$ already have the conventions expected by their factors.

In [13]:
graph = gtsam.NonlinearFactorGraph()
exact = gtsam.Values()

# Matrix-weighted states are kTw, the inverse of GTSAM ground truth wTk.
for k, wTk in enumerate(example.wTks):
    exact.insert(X(k), wTk.inverse())

for k, wL, kP, information in measurements.landmarks:
    graph.add(
        gtsam.KnownLandmarkFactorPose3(
            X(k),
            wL,
            kP,
            gtsam.noiseModel.Gaussian.Information(information),
        )
    )

odometry_model = gtsam.noiseModel.Isotropic.Sigma(6, ODOMETRY_SIGMA)
for i, j, iTj in measurements.odometry:
    graph.add(
        gtsam.FrobeniusLeftBetweenFactorPose3(
            X(i), X(j), iTj, odometry_model
        )
    )

first_information = measurements.landmarks[0].information
covariance_eigenvalues = np.linalg.eigvalsh(np.linalg.inv(first_information))
print(f"factors: {graph.size()}")
print("covariance eigenvalues:", covariance_eigenvalues)
print("sqrt condition number:", np.sqrt(covariance_eigenvalues[-1] / covariance_eigenvalues[0]))

factors: 14
covariance eigenvalues: [1.e-04 1.e-04 1.e-02]
sqrt condition number: 10.00000000000005


## Perturb and optimize

Initialization must use the graph's convention. For each conventional ground-truth pose $wTk$, we first compute $kTw=(wTk)^{-1}$ and then retract the perturbation from that inverse pose. Perturbing $wTk$ directly would initialize a different variable.

In [14]:
perturbation = np.array([0.02, -0.015, 0.01, 0.08, -0.05, 0.06])
initial = gtsam.Values()
for k, wTk in enumerate(example.wTks):
    kTw = wTk.inverse()
    initial.insert(X(k), kTw.retract((k + 1) * perturbation))

parameters = gtsam.GaussNewtonParams()
parameters.setMaxIterations(100)
parameters.setRelativeErrorTol(1e-12)
result = gtsam.GaussNewtonOptimizer(
    graph, initial, parameters
).optimize()

exact_error = graph.error(exact)
initial_error = graph.error(initial)
final_error = graph.error(result)
pose_errors = np.array([
    np.linalg.norm(
        wTk.localCoordinates(result.atPose3(X(k)).inverse())
    )
    for k, wTk in enumerate(example.wTks)
])

print(f"exact graph error:  {exact_error:.3e}")
print(f"initial error:      {initial_error:.3e}")
print(f"final error:        {final_error:.3e}")
print(f"mean pose error:    {pose_errors.mean():.3e}")
print(f"maximum pose error: {pose_errors.max():.3e}")

assert exact_error < 1e-8
assert final_error < 1e-8
assert pose_errors.max() < 1e-8

exact graph error:  5.697e-27
initial error:      6.694e+03
final error:        5.673e-20
mean pose error:    6.317e-13
maximum pose error: 1.786e-12


In [15]:
# Ground truth is already wTk; graph values must be inverted back from kTw.
ground_truth_wPs = np.vstack([wTk.translation() for wTk in example.wTks])
initial_wPs = np.vstack(
    [initial.atPose3(X(k)).inverse().translation() for k in range(example.num_poses)]
)
result_wPs = np.vstack(
    [result.atPose3(X(k)).inverse().translation() for k in range(example.num_poses)]
)
wLs_array = np.vstack(example.wLs)

fig = go.Figure()
fig.add_scatter(
    x=initial_wPs[:, 0], y=initial_wPs[:, 1], mode="lines+markers",
    line={"dash": "dash"}, name="initial poses",
)
fig.add_scatter(
    x=ground_truth_wPs[:, 0], y=ground_truth_wPs[:, 1],
    mode="lines+markers", line={"color": "black"}, name="ground truth",
)
fig.add_scatter(
    x=result_wPs[:, 0], y=result_wPs[:, 1], mode="lines+markers",
    marker={"symbol": "x"}, name="optimized poses",
)
fig.add_scatter(
    x=wLs_array[:, 0], y=wLs_array[:, 1], mode="markers",
    marker={"symbol": "star", "size": 12}, name="known landmarks",
)
fig.update_layout(
    title="Matrix-weighted localization",
    xaxis_title="world x [m]", yaxis_title="world y [m]",
    template="plotly_white", width=700, height=500,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

## Reading the result

The covariance eigenvalues should be $\sigma_l^2,\sigma_l^2,\sigma_r^2$, so their square-root condition number is the requested anisotropicity. The full information matrix is generally not diagonal in sensor coordinates because its radial eigenvector follows the observation ray.

The optimizer returns $kTw$ states, whereas the dataset stores $wTk$. We therefore invert every estimated pose before evaluating its local-coordinate error against ground truth. The noiseless ground truth has zero objective, and Gauss--Newton returns the perturbed initialization to that solution.